# Week 17: Production Retrieval-Augmented Generation (Advanced RAG) & Vector Search
## Notebook 03: 2-Stage Retrieval: Bi-Encoders, Cross-Encoders & ColBERT Late Interaction

### 1. Conceptual Intuition & Real-World Use Case
In production RAG systems, there is an inescapable trade-off between **retrieval latency** and **relevance quality**:
- **Bi-Encoders (e.g. OpenAI `text-embedding-3`, BGE-small):** Compute embeddings for query and passages independently. Query-passage interaction happens via a cheap dot product. *Extremely fast ($<10$ms) over 10M vectors, but misses intricate token-level interactions.*
- **Cross-Encoders (e.g. Cohere Rerank, `bge-reranker-large`):** Feeds `[CLS] Query [SEP] Passage [SEP]` jointly into full multi-layer transformer cross-attention. *Massively higher accuracy, but $O(L^2)$ attention makes it too slow to run on entire databases.*

**Industry Solution: 2-Stage Retrieval Pipeline:**
$$\text{Corpus (1,000,000 docs)} \xrightarrow[\text{Fast Bi-Encoder}]{Top-100} \text{Candidates} \xrightarrow[\text{Deep Cross-Encoder / ColBERT}]{Top-5} \text{LLM Context}$$

---

### 2. Mathematical Foundations: ColBERT Late Interaction
Introduced by Khattab & Zaharia (Stanford), **ColBERT** bridges the speed of Bi-Encoders with the accuracy of Cross-Encoders using **Late Interaction**.

1. Embed all query tokens: $E_q = \{e_{q,1}, \dots, e_{q,l}\}$.
2. Embed all document tokens: $E_d = \{e_{d,1}, \dots, e_{d,m}\}$.
3. Compute **MaxSim Operator**:
   $$\text{Score}(q, d) = \sum_{i=1}^{|Q|} \max_{j=1}^{|D|} \left( e_{q,i} \cdot e_{d,j}^T \right)$$
   For every token in the query, find the single highest cosine similarity across all tokens in the document, and sum them. This captures fine-grained token alignments with pre-computable offline document embeddings!

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Simulate Token Embeddings for Query and 2 Candidate Passages
dim = 8  # Embedding dimension per token

# Query: "Python GIL multithreading bottleneck" (4 tokens)
query_tokens = ["python", "gil", "multithreading", "bottleneck"]
E_query = np.random.randn(len(query_tokens), dim)
E_query /= np.linalg.norm(E_query, axis=1, keepdims=True)

# Passage A: Relevant technical explanation
# "Global Interpreter Lock prevents concurrent thread execution in Python"
E_doc_A = np.random.randn(8, dim)
# Make specific tokens strongly align with query concepts
E_doc_A[0] = E_query[1] + 0.1 * np.random.randn(dim) # aligns with GIL
E_doc_A[4] = E_query[2] + 0.1 * np.random.randn(dim) # aligns with multithreading
E_doc_A /= np.linalg.norm(E_doc_A, axis=1, keepdims=True)

# Passage B: Irrelevant passage about biology
# "The python snake exhibits predatory behavior in subtropical habitats"
E_doc_B = np.random.randn(8, dim)
E_doc_B[0] = E_query[0] + 0.1 * np.random.randn(dim) # aligns only with python keyword
E_doc_B /= np.linalg.norm(E_doc_B, axis=1, keepdims=True)

# Step 2: Implement ColBERT Late Interaction MaxSim
def colbert_maxsim(E_q, E_d):
    # Compute token-to-token similarity matrix: (len_q, len_d)
    sim_matrix = E_q @ E_d.T
    # For each query token, take maximum similarity across document tokens
    max_per_query_token = np.max(sim_matrix, axis=1)
    # Sum over all query tokens
    return np.sum(max_per_query_token), sim_matrix

score_A, sim_mat_A = colbert_maxsim(E_query, E_doc_A)
score_B, sim_mat_B = colbert_maxsim(E_query, E_doc_B)

print("--- ColBERT Late Interaction MaxSim Scores ---")
print(f"Passage A (Technical GIL Explanation): {score_A:.4f}")
print(f"Passage B (Python Snake Biology):       {score_B:.4f}")

# Step 3: Compare with Naive Single-Vector Mean Pooling (Bi-Encoder)
vec_q = np.mean(E_query, axis=0)
vec_A = np.mean(E_doc_A, axis=0)
vec_B = np.mean(E_doc_B, axis=0)

cos_A = np.dot(vec_q, vec_A) / (np.linalg.norm(vec_q) * np.linalg.norm(vec_A))
cos_B = np.dot(vec_q, vec_B) / (np.linalg.norm(vec_q) * np.linalg.norm(vec_B))

print("\n--- Naive Mean-Pooled Bi-Encoder Cosine Scores ---")
print(f"Passage A Cosine: {cos_A:.4f}")
print(f"Passage B Cosine: {cos_B:.4f}")

### 3. Step-by-Step Code Walkthrough
1. **Token Matrix Multiplication:** `E_q @ E_d.T` produces a matrix of all pairwise token cosine similarities.
2. **MaxSim Operator:** `np.max(sim_matrix, axis=1)` identifies the best matching document token for each query word.
3. **Discriminative Power:** Passage B matches "python" but completely misses "gil", "multithreading", and "bottleneck". ColBERT rewards Passage A because multiple key concepts find matching tokens.

---

### 4. Industry Applications & Trade-offs
- **Cohere Rerank:** Used in enterprise RAG pipelines to filter Top-100 vector hits down to Top-5 highest quality chunks.
- **Latency Budget:** Bi-encoder retrieval takes $\sim 5$ms; Cross-encoder reranking of 50 chunks takes $\sim 50$ms.

---

### 5. Top Interview Questions & Answers
**Q1: Why is Cross-Encoder reranking too computationally expensive to use for initial candidate retrieval?**
> **Answer:** Cross-encoders require concatenating the query with every passage and running full multi-head transformer cross-attention. For 1,000,000 passages, that requires 1,000,000 forward transformer passes per query ($O(N \cdot L^2)$), which is impossible for interactive latency.

**Q2: How does ColBERT achieve near Cross-Encoder quality at Bi-Encoder speeds?**
> **Answer:** ColBERT pre-computes and stores all document token embeddings offline. At query time, only the query tokens are embedded, and scoring reduces to fast matrix multiplication and max-pooling (MaxSim), bypassing joint Transformer cross-attention layers during retrieval.